In [1]:
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
import re # standard library to use regex
import ast
import fasttext # library for language detection
from IPython.display import display
from pathlib import Path

In [2]:
# Configuration; centralized source of common variables 
TEXT_COLS = ["short_description", "about_the_game", "detailed_description"]
LABEL_COL = "genres"
REL_COLS = ["appid", "name", "detailed_description", "about_the_game", "short_description", "genres"]
CONTENT_GENRES = ["Action", "Adventure", "Massively Multiplayer", "Racing", "RPG", "Simulation", "Sports", "Strategy"]
ADULT_GENRES = ["Nudity", "Sexual Content", "Gore"]
LID_MODEL = fasttext.load_model("lid.176.bin")
SEED = 42
TESTSET_SIZE = 1000
VALSET_SIZE = 1000
OUTPUT_DIR = Path("Data")
step_log = []

# Data Pre-Processing

In [3]:
# read in the dataset
df = pd.read_csv(r'data/games_march2025_cleaned.csv')
df

,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,730,Counter-Strike 2,2012-08-21,0,0.00,1,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",NaN,...,879,5174,350,0,1212356,"{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",86,8632939,82,96473
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0,0.00,0,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,NaN,...,0,0,0,0,616738,"{'Survival': 14838, 'Shooter': 12727, 'Battle ...",59,2513842,68,16720
2,570,Dota 2,2013-07-09,0,0.00,2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...,...,1536,898,892,0,555977,"{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",81,2452595,80,29366
3,271590,Grand Theft Auto V Legacy,2015-04-13,17,0.00,0,"When a young street hustler, a retired bank ro...","When a young street hustler, a retired bank ro...",Grand Theft Auto V for PC offers players the o...,NaN,...,771,7101,74,0,117698,"{'Open World': 32644, 'Action': 23539, 'Multip...",87,1803832,92,17517
4,359550,Tom Clancy's Rainbow Six® Siege,2015-12-01,17,3.99,9,Edition Comparison Ultimate Edition The Tom Cl...,“One of the best first-person shooters ever ma...,"Tom Clancy's Rainbow Six® Siege is an elite, t...",NaN,...,682,2434,306,80,89916,"{'FPS': 9831, 'PvP': 9162, 'e-sports': 9072, '...",84,1168020,76,12608
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89613,2115110,Outrun Them,2024-02-29,0,3.99,0,The name of the game is literally the game. Ou...,The name of the game is literally the game. Ou...,It's all in the name.. outrun them.,NaN,...,0,0,0,0,0,"{'2D Platformer': 91, 'Runner': 85, 'Platforme...",-1,-1,-1,-1
89614,1174200,Lands of Pharaoh: Episode 1,2020-01-07,0,10.00,0,Join Our Discord Channel About the Game Briefl...,Briefly: Lands of Pharaoh is a third person vi...,Lands of Pharaoh is a multi-camera-angle based...,NaN,...,0,0,0,0,0,"{'Action': 43, 'Adventure': 41, 'Indie': 41, '...",-1,-1,-1,-1
89615,1160190,Player One,2019-10-10,0,1.99,0,Player One is a new 3D game where you take con...,Player One is a new 3D game where you take con...,"3D Game, Thirdperson Platformer",NaN,...,0,0,0,0,0,"{'Indie': 31, 'Early Access': 21}",-1,-1,-1,-1
89616,3380340,DragonRoad,2025-01-25,0,29.99,0,"This is an open world game on 3D, also a m...","This is an open world game on 3D, also a m...","This is an open world game on 3D, Also a missi...",NaN,...,0,0,0,0,0,"{'RPG': 117, 'Action-Adventure': 111, 'Strateg...",-1,-1,-1,-1


## Descriptive Measures

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89618 entries, 0 to 89617
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   appid                     89618 non-null  int64  
 1   name                      89618 non-null  object 
 2   release_date              89618 non-null  object 
 3   required_age              89618 non-null  int64  
 4   price                     89618 non-null  float64
 5   dlc_count                 89618 non-null  int64  
 6   detailed_description      89421 non-null  object 
 7   about_the_game            89398 non-null  object 
 8   short_description         89498 non-null  object 
 9   reviews                   10401 non-null  object 
 10  header_image              89618 non-null  object 
 11  website                   41114 non-null  object 
 12  support_url               44110 non-null  object 
 13  support_email             78798 non-null  object 
 14  window

In [5]:
df.describe()

,appid,required_age,price,dlc_count,metacritic_score,achievements,recommendations,user_score,score_rank,positive,...,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
count,8.961800e+04,89618.000000,89618.000000,89618.000000,89618.000000,89618.000000,8.961800e+04,89618.000000,39.000000,8.961800e+04,...,8.961800e+04,89618.000000,8.961800e+04,89618.000000,89618.000000,8.961800e+04,89618.000000,8.961800e+04,89618.000000,89618.000000
mean,1.656904e+06,0.183624,7.309623,0.595583,2.903245,20.552333,1.009401e+03,0.032817,99.128205,1.269896e+03,...,1.149119e+02,5.032694,1.147559e+02,5.301814,4.558627,9.833994e+01,45.352418,1.315490e+03,5.231639,15.584581
std,9.168390e+05,1.725594,13.331073,15.351920,14.445358,163.562418,2.204815e+04,1.615149,0.695076,3.181441e+04,...,6.814748e+03,181.100334,8.806711e+03,189.591095,16.538036,5.717544e+03,40.685695,3.542370e+04,22.335358,444.277307
min,2.000000e+01,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,98.000000,0.000000e+00,...,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,-1.000000,-1.000000e+00,-1.000000,-1.000000
25%,8.550525e+05,0.000000,0.990000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,99.000000,1.000000e+00,...,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,-1.000000,-1.000000e+00,-1.000000,-1.000000
50%,1.524730e+06,0.000000,4.990000,0.000000,0.000000,5.000000,0.000000e+00,0.000000,99.000000,1.000000e+01,...,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,60.000000,1.500000e+01,-1.000000,-1.000000
75%,2.430852e+06,0.000000,9.990000,0.000000,0.000000,20.000000,0.000000e+00,0.000000,100.000000,5.800000e+01,...,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,84.000000,8.100000e+01,-1.000000,-1.000000
max,3.542350e+06,21.000000,999.980000,3427.000000,97.000000,9821.000000,4.401572e+06,100.000000,100.000000,7.480813e+06,...,1.462997e+06,18568.000000,1.462997e+06,18568.000000,100.000000,1.212356e+06,100.000000,8.632939e+06,100.000000,96473.000000


In [6]:
df["genres"].head()

0                           ['Action', 'Free To Play']
1    ['Action', 'Adventure', 'Massively Multiplayer...
2               ['Action', 'Strategy', 'Free To Play']
3                              ['Action', 'Adventure']
4                                           ['Action']
Name: genres, dtype: object

In [7]:
duplicates_by_columns = df[df.duplicated(subset=["appid", "name"])]
duplicates_by_columns

,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent


## Reduce to relevant columns
The first pre-processing step involves reducing the original dataset to a preselection of relevant columns. This comprises appid, name, detailed_description, about_the_game, short_description, categories, genres, and tags. Any columns beyond these were not included in this subset.

In [8]:
# Create a reduced subset that only contains relevant columns
df = df.loc[:, REL_COLS]
step_log.append({"step": "1 | Variable selection (no records deleted)", "n_rows": len(df)})
df

,appid,name,detailed_description,about_the_game,short_description,genres
0,730,Counter-Strike 2,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","['Action', 'Free To Play']"
1,578080,PUBG: BATTLEGROUNDS,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,"['Action', 'Adventure', 'Massively Multiplayer..."
2,570,Dota 2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...","['Action', 'Strategy', 'Free To Play']"
3,271590,Grand Theft Auto V Legacy,"When a young street hustler, a retired bank ro...","When a young street hustler, a retired bank ro...",Grand Theft Auto V for PC offers players the o...,"['Action', 'Adventure']"
4,359550,Tom Clancy's Rainbow Six® Siege,Edition Comparison Ultimate Edition The Tom Cl...,“One of the best first-person shooters ever ma...,"Tom Clancy's Rainbow Six® Siege is an elite, t...",['Action']
...,...,...,...,...,...,...
89613,2115110,Outrun Them,The name of the game is literally the game. Ou...,The name of the game is literally the game. Ou...,It's all in the name.. outrun them.,"['Indie', 'Early Access']"
89614,1174200,Lands of Pharaoh: Episode 1,Join Our Discord Channel About the Game Briefl...,Briefly: Lands of Pharaoh is a third person vi...,Lands of Pharaoh is a multi-camera-angle based...,"['Action', 'Adventure', 'Indie', 'Strategy']"
89615,1160190,Player One,Player One is a new 3D game where you take con...,Player One is a new 3D game where you take con...,"3D Game, Thirdperson Platformer","['Indie', 'Early Access']"
89616,3380340,DragonRoad,"This is an open world game on 3D, also a m...","This is an open world game on 3D, also a m...","This is an open world game on 3D, Also a missi...","['Adventure', 'RPG', 'Simulation', 'Strategy']"


## Clean HTML
In this step, the relevant text based columns 'short_description', 'about_the_game', and 'detailed_description' are cleaned from HTML code as well as inserted images. As they dont give any meaningful input for the LLM in the following experiment and thus just bloat the amount of input tokens needed. 

In [9]:
def clean_html(text):
    if pd.isna(text):
        return np.nan
        
    soup = BeautifulSoup(text, "html.parser")
    for tag in soup(["img", "video", "source", "script", "style"]):
        tag.decompose()                           # delete media and code in text based columns
    text = soup.get_text(separator="\n")          # Tags weg, Entities dekodiert
    text = re.sub(r"[ \t\xa0]+", " ", text)       # Leerzeichen in Zeilen zusammenfassen
    text = re.sub(r"\s*\n\s*", "\n", text)        # Leerzeilen zusammenfassen
    return text.strip()

In [10]:
for col in TEXT_COLS:
    df[col] = df[col].apply(clean_html)
step_log.append({"step": "2 | HTML Cleanup (no records deleted)", "n_rows": len(df)})

In [11]:
df

,appid,name,detailed_description,about_the_game,short_description,genres
0,730,Counter-Strike 2,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","['Action', 'Free To Play']"
1,578080,PUBG: BATTLEGROUNDS,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,"['Action', 'Adventure', 'Massively Multiplayer..."
2,570,Dota 2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...","['Action', 'Strategy', 'Free To Play']"
3,271590,Grand Theft Auto V Legacy,"When a young street hustler, a retired bank ro...","When a young street hustler, a retired bank ro...",Grand Theft Auto V for PC offers players the o...,"['Action', 'Adventure']"
4,359550,Tom Clancy's Rainbow Six® Siege,Edition Comparison Ultimate Edition The Tom Cl...,“One of the best first-person shooters ever ma...,"Tom Clancy's Rainbow Six® Siege is an elite, t...",['Action']
...,...,...,...,...,...,...
89613,2115110,Outrun Them,The name of the game is literally the game. Ou...,The name of the game is literally the game. Ou...,It's all in the name.. outrun them.,"['Indie', 'Early Access']"
89614,1174200,Lands of Pharaoh: Episode 1,Join Our Discord Channel About the Game Briefl...,Briefly: Lands of Pharaoh is a third person vi...,Lands of Pharaoh is a multi-camera-angle based...,"['Action', 'Adventure', 'Indie', 'Strategy']"
89615,1160190,Player One,Player One is a new 3D game where you take con...,Player One is a new 3D game where you take con...,"3D Game, Thirdperson Platformer","['Indie', 'Early Access']"
89616,3380340,DragonRoad,"This is an open world game on 3D, also a missi...","This is an open world game on 3D, also a missi...","This is an open world game on 3D, Also a missi...","['Adventure', 'RPG', 'Simulation', 'Strategy']"


## Drop missing values
Based on the HTML-cleaned data set, all empty or missing entries are now filtered out. This applies to both the TEXT_COL and the LABEL_COL. Generating synthetic or averaged values to fill in the gaps does not make sense in this context.

In [12]:
s = pd.read_csv(r"data/games_march2025_cleaned.csv", usecols=["genres"])["genres"]

print(s.head(10).to_string())
print("Missing (NaN):", s.isna().sum())
print("Proportion in list format:", s.dropna().str.strip().str.startswith("[").mean())

# Check that every instance can be parsed as a list
def parses(x):
    try:
        return isinstance(ast.literal_eval(x), list)
    except (ValueError, SyntaxError):
        return False
print("Can not be parsed:", (~s.dropna().apply(parses)).sum())

0                           ['Action', 'Free To Play']
1    ['Action', 'Adventure', 'Massively Multiplayer...
2               ['Action', 'Strategy', 'Free To Play']
3                              ['Action', 'Adventure']
4                                           ['Action']
5                           ['Action', 'Free To Play']
6              ['Action', 'Adventure', 'Indie', 'RPG']
7    ['Action', 'Adventure', 'Indie', 'Massively Mu...
8                    ['Casual', 'Indie', 'Simulation']
9              ['Action', 'Adventure', 'Free To Play']
Missing (NaN): 0
Proportion in list format: 1.0
Can not be parsed: 0


In [13]:
for col in TEXT_COLS:
    df[col] = df[col].where(df[col].str.strip().astype(bool), np.nan)
 
df[LABEL_COL] = df[LABEL_COL].fillna("[]").apply(ast.literal_eval)
df = df.dropna(subset=TEXT_COLS)
df = df[df[LABEL_COL].str.len() > 0]
step_log.append({"step": "3 | Remove missing values", "n_rows": len(df)})
 
print("\nAll genres featured (for checking the label set):")
print(df[LABEL_COL].explode().value_counts().to_string(), "\n")


All genres featured (for checking the label set):
genres
Indie                    63110
Casual                   38657
Action                   36780
Adventure                35394
Simulation               18548
Strategy                 17338
RPG                      16314
Early Access              9097
Free To Play              8848
Sports                    3932
Racing                    3296
Massively Multiplayer     2113
Utilities                  856
Design & Illustration      494
Violent                    429
Animation & Modeling       394
Education                  389
Video Production           285
Gore                       265
Game Development           256
Audio Production           195
Software Training          191
Photo Editing              149
Web Publishing              99
Nudity                      96
Sexual Content              90
Accounting                  24
Movie                        2
Documentary                  1
Episodic                     1
Short       

In [14]:
assert df[TEXT_COLS].notna().all().all(), "Descriptions still missing."
assert (df[TEXT_COLS].apply(lambda s: s.str.strip().str.len()) > 0).all().all(), "Descriptions still missing."
assert (df[LABEL_COL].str.len() > 0).all(), "Empty genres remain."
print("All checks passed.")

All checks passed.


## Check for adult content

In [15]:
has_genre = df[LABEL_COL].apply(lambda gs: bool(set(gs).intersection(ADULT_GENRES)))
df = df[~has_genre].copy()
step_log.append({"step": "4 | Remove adult content", "n_rows": len(df)})

## Language filter (check for english only)

In [16]:
LANG_MIN_PROB = {"short_description": 0.85,
                 "about_the_game": 0.85,
                 "detailed_description": 0.85}

In [17]:
def text_chunks(text: str, chunk: int = 1000) -> list[str]:
    """Keep short texts in their entirety and split long texts in the beginning, middle and end."""
    if len(text) <= chunk:
        return [text]
    mid = len(text) // 2
    return [text[:chunk], text[mid - chunk // 2 : mid + chunk // 2], text[-chunk:]]

def is_english(series: pd.Series, min_prob: float) -> pd.Series:
    """True, if all sections of a text are in English with a probability of >= min_prob."""
    chunks = [text_chunks(t) for t in series]
    flat = [c.replace("\n", " ") for cs in chunks for c in cs]   # remove escape sequence as fasttext requires single-line texts
    labels, probs = LID_MODEL.predict(flat, k=1)
    en_prob = [p[0] if l[0] == "__label__en" else 0.0 for l, p in zip(labels, probs)]

    result, i = [], 0
    for cs in chunks:
        result.append(all(p >= min_prob for p in en_prob[i : i + len(cs)]))
        i += len(cs)
    return pd.Series(result, index=series.index)

lang_ok = pd.DataFrame({c: is_english(df[c], p) for c, p in LANG_MIN_PROB.items()})
print("Non-English text per column:")

display((~lang_ok).sum().rename("Count").to_frame())
print("Disagreement between the columns:", (lang_ok.nunique(axis=1) > 1).sum())

# Review of the rejected texts for verification
rejected = df[~lang_ok.all(axis=1)].copy()

# In which column of the game did the language filter fail?
rejected["failed_cols"] = (~lang_ok.loc[rejected.index]).apply(
    lambda r: ", ".join(r.index[r]), axis=1)

def worst_chunk(text: str):
    """Section with the lowest P(en) (probability that the text is english): recognised language, probability, text extract."""
    chunks = [c.replace("\n", " ") for c in text_chunks(text)]
    labels, probs = LID_MODEL.predict(chunks, k=1)
    en = [p[0] if l[0] == "__label__en" else 0.0 for l, p in zip(labels, probs)]
    i = int(np.argmin(en))
    return labels[i][0].replace("__label__", ""), round(float(probs[i][0]), 3), chunks[i][:300]

# Sample table with details for each column
sample = rejected.sample(min(30, len(rejected)), random_state=SEED)
for col in TEXT_COLS:
    sample[[f"{col}_lang", f"{col}_prob", f"{col}_excerpt"]] = (
        sample[col].apply(lambda t: pd.Series(worst_chunk(t))))

pd.set_option("display.max_colwidth", 300)
for col in TEXT_COLS:
    print(f"\n=== {col} ===")
    display(sample.loc[sample["failed_cols"].str.contains(col),
                       ["name", "failed_cols", f"{col}_lang", f"{col}_prob", f"{col}_excerpt"]])

# Overview: Which languages were detected? (by column, failed texts only)
for col in TEXT_COLS:
    failed = rejected[rejected["failed_cols"].str.contains(col)]
    langs = failed[col].apply(lambda t: worst_chunk(t)[0])
    print(f"\nErkannte Sprachen in {col} (n = {len(failed)}):")
    display(langs.value_counts().head(10).rename("Count").to_frame())

# Optional: export everything, e.g. to view in Excel
# rejected[["appid", "name", "failed_cols", *TEXT_COLS]].to_csv(
    # OUTPUT_DIR / "rejected_language.csv", index=False, encoding="utf-8-sig")

df = df[lang_ok.all(axis=1)]
step_log.append({"step": "5 | Language filter (english)", "n_rows": len(df)})

Non-English text per column:


,Count
short_description,14679
about_the_game,6863
detailed_description,7248


Disagreement between the columns: 15156

=== short_description ===


,name,failed_cols,short_description_lang,short_description_prob,short_description_excerpt
15204,Around the World in 80 days: Hidden Objects,short_description,en,0.816,You will travel round the world with the Jules Verne heroes! Each location (scene) is a new continent or a new country where you will find yourself in the atmosphere of this region thanks to the picturesque and colorful graphics and the unique musical background.
41263,Maxi Pool Masters VR,short_description,en,0.840,Maxi pool Masters takes you inside the action in a fun and innovative pool game.
53182,Devils Adventure,short_description,en,0.808,2.5d Platformer where you control devil character. Mission to avoid traps and enemies and finish different levels and find skulls
10604,Neckbreak,short_description,en,0.842,"An overdose of reality can have dire consequences... Tumble down the rabbit hole and embrace your downfall with 12 endings, non linear day by day progression, sanity system and more..."
86741,Endless Furry Asteroids,"short_description, about_the_game, detailed_description",en,0.717,Welcome to Endless Furry Asteroids! The 6th installment of the 'Endless Furry' Series!
11740,The Sushi Spinnery,short_description,en,0.586,Revolutionize the revolving sushi bar in this fresh new sim!
32160,Oppaidius Desert Island!,"short_description, about_the_game, detailed_description",en,0.841,"W-w-w-whaaaaat...? I'm on a desert island, alone with Angela... NAKED...? Discover what happens next in this original visual novel!"
27580,Moon's Creed,short_description,en,0.755,Choose your zodiac hero. Craft unique decks. Collect astral boons. Vanquish corrupted constellations. Ascend through cosmic chaos!
19427,Eldritchvania,short_description,en,0.828,"Explore unearthly ruins, solve arcane riddles and pick up your sword against the indescribable horrors that lurk amidst the forgotten depths of our world. Beware the creeping grip of madness as you uncover secrets not meant for the eyes of men!"
77269,Winterland,short_description,en,0.554,"This is a short peaceful RPG. Your only commands are 'joke', 'dance', 'stare', 'run away', and 'hug'."



=== about_the_game ===


...

,Count
short_description,
en,13011
zh,1059
ja,353
ko,50
ru,33
fr,32
pt,30
es,27
de,26



Erkannte Sprachen in about_the_game (n = 6863):


,Count
about_the_game,
en,5203
zh,1076
ja,341
ko,52
ru,42
fr,27
pt,26
es,24
de,22



Erkannte Sprachen in detailed_description (n = 7248):


,Count
detailed_description,
en,5578
zh,1075
ja,350
ko,52
ru,42
fr,27
pt,26
es,25
de,23


In [18]:
df

,appid,name,detailed_description,about_the_game,short_description,genres
0,730,Counter-Strike 2,"For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2. A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical l...","For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2. A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical l...","For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.","[Action, Free To Play]"
2,570,Dota 2,"The most-played game on Steam. Every day, millions of players worldwide enter battle as one of over a hundred Dota heroes. And no matter if it's their 10th hour of play or 1,000th, there's always something new to discover. With regular updates that ensure a constant evolution of gameplay, featur...","The most-played game on Steam. Every day, millions of players worldwide enter battle as one of over a hundred Dota heroes. And no matter if it's their 10th hour of play or 1,000th, there's always something new to discover. With regular updates that ensure a constant evolution of gameplay, featur...","Every day, millions of players worldwide enter battle as one of over a hundred Dota heroes. And no matter if it's their 10th hour of play or 1,000th, there's always something new to discover. With regular updates that ensure a constant evolution of gameplay, features, and heroes, Dota 2 has take...","[Action, Strategy, Free To Play]"
3,271590,Grand Theft Auto V Legacy,"When a young street hustler, a retired bank robber and a terrifying psychopath find themselves entangled with some of the most frightening and deranged elements of the criminal underworld, the U.S. government and the entertainment industry, they must pull off a series of dangerous heists to surv...","When a young street hustler, a retired bank robber and a terrifying psychopath find themselves entangled with some of the most frightening and deranged elements of the criminal underworld, the U.S. government and the entertainment industry, they must pull off a series of dangerous heists to surv...","Grand Theft Auto V for PC offers players the option to explore the award-winning world of Los Santos and Blaine County in resolutions of up to 4k and beyond, as well as the chance to experience the game running at 60 frames per second.","[Action, Adventure]"
5,440,Team Fortress 2,"'The most fun you can have online' - PC Gamer Is now FREE! There’s no catch! Play as much as you want, as long as you like! The most highly-rated free game of all time! One of the most popular online action games of all time, Team Fortress 2 delivers constant free updates—new game modes, maps, e...","'The most fun you can have online' - PC Gamer Is now FREE! There’s no catch! Play as much as you want, as long as you like! The most highly-rated free game of all time! One of the most popular online action games of all time, Team Fortress 2 delivers constant free updates—new game modes, maps, e...","Nine distinct classes provide a broad range of tactical abilities and personalities. Constantly updated with new game modes, maps, equipment and, most importantly, hats!","[Action, Free To Play]"
6,105600,Terraria,"Dig, Fight, Explore, Build: The very world is at your fingertips as you fight for survival, fortune, and glory. Will you delve deep into cavernous expanses in search of treasure and raw materials with which to craft ever-evolving gear, machinery, and aesthetics? Perhaps you will choose instead t...","Dig, Fight, Explore, Build: The very world is at yo

## Reduce the labelset to content genres
In this step, the range of genres is reduced to a predefined set. This involves, for example, removing all non-games like software, as well as rather undefined and unclear genres such as Early Access or Free to Play which do not provide any indication of the game’s content or nature. 

In [19]:
df[LABEL_COL] = df[LABEL_COL].apply(lambda gs: sorted(set(gs).intersection(CONTENT_GENRES)))
df = df[df[LABEL_COL].str.len() > 0]
step_log.append({"step": "6 | Reduce set of labels", "n_rows": len(df)})

df

,appid,name,detailed_description,about_the_game,short_description,genres
0,730,Counter-Strike 2,"For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2. A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical l...","For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2. A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical l...","For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.",[Action]
2,570,Dota 2,"The most-played game on Steam. Every day, millions of players worldwide enter battle as one of over a hundred Dota heroes. And no matter if it's their 10th hour of play or 1,000th, there's always something new to discover. With regular updates that ensure a constant evolution of gameplay, featur...","The most-played game on Steam. Every day, millions of players worldwide enter battle as one of over a hundred Dota heroes. And no matter if it's their 10th hour of play or 1,000th, there's always something new to discover. With regular updates that ensure a constant evolution of gameplay, featur...","Every day, millions of players worldwide enter battle as one of over a hundred Dota heroes. And no matter if it's their 10th hour of play or 1,000th, there's always something new to discover. With regular updates that ensure a constant evolution of gameplay, features, and heroes, Dota 2 has take...","[Action, Strategy]"
3,271590,Grand Theft Auto V Legacy,"When a young street hustler, a retired bank robber and a terrifying psychopath find themselves entangled with some of the most frightening and deranged elements of the criminal underworld, the U.S. government and the entertainment industry, they must pull off a series of dangerous heists to surv...","When a young street hustler, a retired bank robber and a terrifying psychopath find themselves entangled with some of the most frightening and deranged elements of the criminal underworld, the U.S. government and the entertainment industry, they must pull off a series of dangerous heists to surv...","Grand Theft Auto V for PC offers players the option to explore the award-winning world of Los Santos and Blaine County in resolutions of up to 4k and beyond, as well as the chance to experience the game running at 60 frames per second.","[Action, Adventure]"
5,440,Team Fortress 2,"'The most fun you can have online' - PC Gamer Is now FREE! There’s no catch! Play as much as you want, as long as you like! The most highly-rated free game of all time! One of the most popular online action games of all time, Team Fortress 2 delivers constant free updates—new game modes, maps, e...","'The most fun you can have online' - PC Gamer Is now FREE! There’s no catch! Play as much as you want, as long as you like! The most highly-rated free game of all time! One of the most popular online action games of all time, Team Fortress 2 delivers constant free updates—new game modes, maps, e...","Nine distinct classes provide a broad range of tactical abilities and personalities. Constantly updated with new game modes, maps, equipment and, most importantly, hats!",[Action]
6,105600,Terraria,"Dig, Fight, Explore, Build: The very world is at your fingertips as you fight for survival, fortune, and glory. Will you delve deep into cavernous expanses in search of treasure and raw materials with which to craft ever-evolving gear, machinery, and aesthetics? Perhaps you will choose instead t...","Dig, Fight, Explore, Build: The very world is at your fingertips as you fight for survival, fortu

## Length of Description Levels

In [20]:
wc = pd.DataFrame({c: df[c].str.split().str.len() for c in TEXT_COLS})

stats = wc.describe(percentiles=[.01, .02, .05, .10, .25, .50, .75, .90, .99]).T
display(stats.style.format("{:.0f}"))

# How many texts fall below the possible limits?
candidates = [3, 5, 10, 15, 20, 30, 50]
display(pd.DataFrame({c: [(wc[c] < t).sum() for t in candidates] for c in TEXT_COLS},
                     index=[f"< {t} words" for t in candidates]))

# What do texts of a certain length look like?
def show_examples(col: str, lo: int, hi: int, n: int = 5):
    sub = df[(wc[col] >= lo) & (wc[col] < hi)]
    print(f"\n=== {col}: {lo}–{hi - 1} words (n = {len(sub)}) ===")
    if sub.empty:
        print("No records.")
        return
    sample = sub.sample(min(n, len(sub)), random_state=SEED)
    table = pd.DataFrame({
        "Game": sample["name"],
        "Words": wc.loc[sample.index, col],
        "Text": sample[col].str.replace("\n", " ").str.slice(0, 300),
    })
    display(table.style
            .hide(axis="index")
            .set_properties(subset=["Text"], **{"text-align": "left", "white-space": "normal"}))

show_examples("short_description", 1, 5)
show_examples("short_description", 5, 10)
show_examples("about_the_game", 1, 10)
show_examples("about_the_game", 10, 30)

,count,mean,std,min,1%,2%,5%,10%,25%,50%,75%,90%,99%,max
short_description,60121,35,12,1,7,9,13,18,27,37,45,50,57,69
about_the_game,60121,224,169,1,40,49,66,83,122,185,278,402,796,11530
detailed_description,60121,228,176,1,40,50,66,84,124,187,283,411,815,11530


,short_description,about_the_game,detailed_description
< 3 words,7,3,2
< 5 words,102,5,4
< 10 words,1240,14,14
< 15 words,3742,37,34
< 20 words,7561,73,68
< 30 words,18705,194,187
< 50 words,53520,1221,1194



=== short_description: 1–4 words (n = 102) ===


Game,Words,Text
MultiVerse,3,A Multiplayer game
Minesweeper Tactics,4,Real-time competitive Minesweeper game
Letter Bender,4,A spelling-based puzzle game.
Drevepsina,4,Combat focused minimalist RTS
Halloween Stories: The Neglected Dead Collector's Edition,4,Sometimes Seeing is Believing!



=== short_description: 5–9 words (n = 1138) ===


Game,Words,Text
Death Taxi 3000,8,Retro arcade racer inspired by the MS-DOS era
Love Accident,8,Friendships begin and end. Everyone has their plans.
Vacation: dayOFF,7,The Destiny of 6-people in the island
xMoon,9,xMoon is a laser-based puzzle game on The Moon.
Maidesktop,6,A pretty maid joins your desktop.



=== about_the_game: 1–9 words (n = 14) ===


Game,Words,Text
none,5,- discontinued - (please remove)
ノンビリア(Nonvilia),6,This game is in Japanese only.
LIMBO,9,"Uncertain of his sister's fate, a boy enters LIMBO"
我们的国,6,In memory of our Homeland's anniversary!
Postponed,1,Postponed



=== about_the_game: 10–29 words (n = 180) ===


Game,Words,Text
不落城-Unconquered Castle,27,Visual Novel + SRPG. The story which happens thousand years after that 'God's one day world'. It's a story telling two brave girls who saved their city.
Pudding,25,This is a game about pudding :) ・Earn pudding while playing! ・Create new pudding! ・Click on the screen! ・Show off your pudding to your friends!
Nerraia,29,"Run, jump, attack and get down. In another World call Nerraia you will be Hura a golden Lion Tamarin that embark in a short-lived adventure to save Nitia island."
4D:XD,16,"Space bullet hell like you've never seen it! Heart pumping music, endless bullets, upgrades, and bosses!"
Frogiee,27,"Frogiee is a fast, precision-based platformer where the main character is Mr. Frogiee. Climb the three mountains and kill the sick and evil frogs. Trailer Music: bensound"


## Check for overlap between description levels

In [21]:
short, about, detailed = df["short_description"], df["about_the_game"], df["detailed_description"]

def share_contained(part: pd.Series, whole: pd.Series) -> float:
    """The proportion of lines in which `part` is fully contained within `whole`."""
    return np.mean([p in w for p, w in zip(part, whole)])
 
print("Overlap between levels of description:")
print(f"  about == detailed:         {(about == detailed).mean():.1%}")
print(f"  short == about:            {(short == about).mean():.1%}")
print(f"  short == detailed:         {(short == detailed).mean():.1%}")

print(f"  short ist Teil von about:     {share_contained(short, about):.1%}")
print(f"  about ist Teil von short:     {share_contained(about, short):.1%}")

print(f"  short ist Teil von detailed:  {share_contained(short, detailed):.1%}")
print(f"  detailed ist Teil von short:  {share_contained(detailed, short):.1%}")

print(f"  detailed ist Teil von about:  {share_contained(detailed, about):.1%}")
print(f"  about ist Teil von detailed:  {share_contained(about, detailed):.1%}\n")

Overlap between levels of description:
  about == detailed:         90.2%
  short == about:            0.5%
  short == detailed:         0.5%
  short ist Teil von about:     22.1%
  about ist Teil von short:     0.5%
  short ist Teil von detailed:  22.1%
  detailed ist Teil von short:  0.5%
  detailed ist Teil von about:  90.2%
  about ist Teil von detailed:  100.0%



In [22]:
# Relative proportions (in terms of word count) of the descriptive levels
wc = {c: df[c].str.split().str.len() for c in TEXT_COLS}
pairs = [("short_description", "about_the_game"),
         ("short_description", "detailed_description"),
         ("about_the_game", "detailed_description")]
pairs = [(a, b) for a, b in pairs if a in TEXT_COLS and b in TEXT_COLS]

rows = []
for a, b in pairs:
    ratio = wc[b] / wc[a]                       # >1: b is longer than a
    b_longer, a_longer = wc[b] > wc[a], wc[a] > wc[b]
    rows.append({
        "Relation": f"{b} / {a}",
        "Q10": ratio.quantile(.10),
        "Median": ratio.median(),
        "Q90": ratio.quantile(.90),
        "Δ Median": (ratio - 1).median(),                                        # (b − a) / a
        "Ø |Δ| Wörter": (wc[b] - wc[a]).abs().mean(),
        "b > a": b_longer.mean(),
        "b larger by": (wc[b][b_longer] / wc[a][b_longer] - 1).median(),          # only where b > a
        "b == a": (wc[b] == wc[a]).mean(),
        "b < a": a_longer.mean(),
        "a larger by": (wc[a][a_longer] / wc[b][a_longer] - 1).median(),          # only where a > b
    })

print("Proportions (word count):")

ratio_table = pd.DataFrame(rows)
display(ratio_table.style
        .hide(axis="index")
        .format({"Q10": "{:.2f}", "Median": "{:.2f}", "Q90": "{:.2f}",
                 "Δ Median": "{:+.1%}",
                 "Ø |Δ| Wörter": "{:.1f}",
                 "b > a": "{:.1%}", "b larger by": "{:.1%}",
                 "b == a": "{:.1%}",
                 "b < a": "{:.1%}", "a larger by": "{:.1%}"},
                na_rep="–"))

Proportions (word count):


Relation,Q10,Median,Q90,Δ Median,Ø |Δ| Wörter,b > a,b larger by,b == a,b < a,a larger by
about_the_game / short_description,2.50,5.54,13.00,+454.1%,188.6,99.2%,457.7%,0.6%,0.2%,20.9%
detailed_description / short_description,2.52,5.61,13.25,+461.3%,192.3,99.2%,464.5%,0.6%,0.2%,20.8%
detailed_description / about_the_game,1.00,1.00,1.00,+0.0%,3.8,9.8%,5.5%,90.2%,0.0%,2.2%


In [23]:
# Drop detailed_description due to similarity with about_the_game
df = df.drop(columns="detailed_description")
TEXT_COLS = [c for c in TEXT_COLS if c != "detailed_description"]
print(f"Remaining description columns: {TEXT_COLS}")

# Remove games where ‘short_description’ and ‘about_the_game’ are fully identical
identical = df["short_description"] == df["about_the_game"]
print(f"short == about: {identical.sum()} Spiele werden entfernt")

df = df[~identical].copy()
step_log.append({"step": "7 | Remove short == about", "n_rows": len(df)})

Remaining description columns: ['short_description', 'about_the_game']
short == about: 310 Spiele werden entfernt


In [24]:
df

,appid,name,about_the_game,short_description,genres
0,730,Counter-Strike 2,"For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2. A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical l...","For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.",[Action]
2,570,Dota 2,"The most-played game on Steam. Every day, millions of players worldwide enter battle as one of over a hundred Dota heroes. And no matter if it's their 10th hour of play or 1,000th, there's always something new to discover. With regular updates that ensure a constant evolution of gameplay, featur...","Every day, millions of players worldwide enter battle as one of over a hundred Dota heroes. And no matter if it's their 10th hour of play or 1,000th, there's always something new to discover. With regular updates that ensure a constant evolution of gameplay, features, and heroes, Dota 2 has take...","[Action, Strategy]"
3,271590,Grand Theft Auto V Legacy,"When a young street hustler, a retired bank robber and a terrifying psychopath find themselves entangled with some of the most frightening and deranged elements of the criminal underworld, the U.S. government and the entertainment industry, they must pull off a series of dangerous heists to surv...","Grand Theft Auto V for PC offers players the option to explore the award-winning world of Los Santos and Blaine County in resolutions of up to 4k and beyond, as well as the chance to experience the game running at 60 frames per second.","[Action, Adventure]"
5,440,Team Fortress 2,"'The most fun you can have online' - PC Gamer Is now FREE! There’s no catch! Play as much as you want, as long as you like! The most highly-rated free game of all time! One of the most popular online action games of all time, Team Fortress 2 delivers constant free updates—new game modes, maps, e...","Nine distinct classes provide a broad range of tactical abilities and personalities. Constantly updated with new game modes, maps, equipment and, most importantly, hats!",[Action]
6,105600,Terraria,"Dig, Fight, Explore, Build: The very world is at your fingertips as you fight for survival, fortune, and glory. Will you delve deep into cavernous expanses in search of treasure and raw materials with which to craft ever-evolving gear, machinery, and aesthetics? Perhaps you will choose instead t...","Dig, fight, explore, build! Nothing is impossible in this action-packed adventure game. Four Pack also available!","[Action, Adventure, RPG]"
...,...,...,...,...,...
89610,2671880,Fortress Defense,"Prepare for an epic battle in Fortress Defense, a thrilling castle wave defense game where you stand as the last line of defense against hordes of relentless enemy skeletons. As the commander of an elite archer squad, your mission is to protect your kingdom from the relentless skeletal onslaught...","As the commander of an elite archer squad, your mission is to protect your kingdom from the relentless skeletal onslaught. Upgrade your archers, unlock potent spells, and discover a vast arsenal of specialized arrows to fend off the unending waves of darkness.","[Action, Adventure, Strategy]"
89611,1295440,Boom slayer,Boom slayer is an reckless and fun arcade. Your mission is shoot skull to reach the goal. But each level will be more complex and challenging !! Every level has limit of shots that player can use to unlock next level. At each level there are several types of opponents. - 18 LEVELS - Different an...,Boom slayer is an reckless and fun arcade. Your mission is shoot skull to reach the goal. But each level will be more complex and challenging !! Every level has limit of shots that player can use to unlock 

## Label Leakage

In [25]:
# REGEX for each genre
GENRE_PATTERNS = {
    "Action":                [r"action"],
    "Adventure":             [r"adventures?"],
    "Massively Multiplayer": [r"massively[- ]multiplayer", r"mmo(?:rpg|rts|fps)?s?"],
    "RPG":                   [r"rpgs?", r"role[- ]playing"],
    "Racing":                [r"racing", r"racers?"],
    "Simulation":            [r"simulations?", r"simulators?"],
    "Sports":                [r"sports?"],
    "Strategy":              [r"strateg(?:y|ies|ic)"],
}
genre_re = {g: re.compile(r"\b(?:" + "|".join(p) + r")\b", re.IGNORECASE)
            for g, p in GENRE_PATTERNS.items()}

# Leakage flag per game and column
for col in TEXT_COLS:
    df[f"leak_{col}"] = [any(genre_re[g].search(t) for g in gs if g in genre_re)
                         for t, gs in zip(df[col], df[LABEL_COL])]

# Leakage rate per split and description column
print("Leakage Rate (Proportion of games whose text mentions a specific ‘gold’ genre):")
display(pd.DataFrame({"Leakage rate": {c: df[f"leak_{c}"].mean() for c in TEXT_COLS}})
        .style.format("{:.1%}"))

rows = []
for g, rx in genre_re.items():
    has_g = df[LABEL_COL].apply(lambda gs: g in gs)
    rows.append({"Genre": g, "n": has_g.sum(),
                 **{c: df.loc[has_g, c].str.contains(rx).mean() for c in TEXT_COLS}})
print("Leakage rate per genre:")
display(pd.DataFrame(rows).style.hide(axis="index").format({c: "{:.1%}" for c in TEXT_COLS}))

step_log.append({"step": "8 | Label Leakage (no records removed)", "n_rows": len(df)})

Leakage Rate (Proportion of games whose text mentions a specific ‘gold’ genre):


,Leakage rate
short_description,29.0%
about_the_game,43.9%


Leakage rate per genre:


Genre,n,short_description,about_the_game
Action,28611,16.5%,26.6%
Adventure,27483,19.7%,34.0%
Massively Multiplayer,1515,15.7%,17.7%
RPG,12222,25.6%,32.6%
Racing,2643,32.4%,42.4%
Simulation,14881,14.9%,23.4%
Sports,3139,10.7%,16.8%
Strategy,13937,21.0%,37.5%


## Split Testset

In [26]:
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [27]:
df = df.reset_index(drop=True)
mlb = MultiLabelBinarizer(classes=sorted(CONTENT_GENRES))
Y = mlb.fit_transform(df[LABEL_COL])

def stratified_split(n: int, Y: np.ndarray, size: int):
    splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=size / n, random_state=SEED)
    return next(splitter.split(np.zeros((n, 1)), Y))

# Split off the test set
rest_idx, test_idx = stratified_split(len(df), Y, TESTSET_SIZE)
df_rest, df_test = df.iloc[rest_idx].reset_index(drop=True), df.iloc[test_idx]
Y_rest = Y[rest_idx]

# Split off the validation set from the remainder
train_idx, val_idx = stratified_split(len(df_rest), Y_rest, VALSET_SIZE)
df_train, df_val = df_rest.iloc[train_idx], df_rest.iloc[val_idx]

# Check for disjunction
ids = {"train": set(df_train["appid"]), "val": set(df_val["appid"]), "test": set(df_test["appid"])}
assert ids["train"].isdisjoint(ids["val"]) and ids["train"].isdisjoint(ids["test"]) \
       and ids["val"].isdisjoint(ids["test"]), "Leakage between the splits."

print("Label distribution (proportion of games by genre):")
display(pd.DataFrame({
    "train": Y_rest[train_idx].mean(axis=0),
    "val":   Y_rest[val_idx].mean(axis=0),
    "test":  Y[test_idx].mean(axis=0),
}, index=mlb.classes_).style.format("{:.3f}"))

print("Leakage rate per data split and description column:")
display(pd.DataFrame({name: {c: part[f"leak_{c}"].mean() for c in TEXT_COLS}
                      for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]})
        .style.format("{:.1%}"))

for i, (name, part) in enumerate([("Train", df_train),
                                  ("Validation", df_val),
                                  ("Test", df_test)], start=1):
    step_log.append({"step": f"8.{i} | {name}", "n_rows": len(part)})
print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

Label distribution (proportion of games by genre):


,train,val,test
Action,0.478,0.478,0.475
Adventure,0.460,0.459,0.456
Massively Multiplayer,0.025,0.025,0.025
RPG,0.204,0.204,0.203
Racing,0.044,0.044,0.044
Simulation,0.249,0.248,0.247
Sports,0.052,0.053,0.053
Strategy,0.233,0.232,0.231


Leakage rate per data split and description column:


,train,val,test
short_description,29.0%,27.5%,29.2%
about_the_game,43.9%,44.2%,43.7%


Train: 57799 | Val: 1003 | Test: 1009


## Save the final datasets & documentation of processing 

In [28]:
df_train.to_parquet(OUTPUT_DIR / "train_set.parquet", index=False)
df_val.to_parquet(OUTPUT_DIR / "val_set.parquet", index=False)
df_test.to_parquet(OUTPUT_DIR / "test_set.parquet", index=False)

pd.DataFrame(step_log).to_csv(OUTPUT_DIR / "preprocessing_log.csv", index=False)
print(f"\nSaved in {OUTPUT_DIR.resolve()}")


Saved in C:\Users\Nils\ICL\data
